In [5]:
import pandas as pd

# loading only small sample first to check the columns and data structure to see if the files have what i need for the project. 
# We will load the full data later after confirming the structure and columns. 

df1 = pd.read_csv("../data/extracted/g_patent.tsv", sep="\t", nrows=5, low_memory=False)
df2 = pd.read_csv("../data/extracted/g_patent_abstract.tsv", sep="\t", nrows=5, low_memory=False)
df3 = pd.read_csv("../data/extracted/g_inventor_disambiguated.tsv", sep="\t", nrows=5, low_memory=False)
df4 = pd.read_csv("../data/extracted/g_assignee_disambiguated.tsv", sep="\t", nrows=5, low_memory=False)
df5 = pd.read_csv("../data/extracted/g_location_disambiguated.tsv",sep="\t", nrows=5, low_memory=False)


print("g_patent columns:")
print(df1.columns.tolist())

print("\ng_patent_abstract columns:")
print(df2.columns.tolist())

print("\ng_inventor_disambiguated columns:")
print(df3.columns.tolist())

print("\ng_assignee_disambiguated columns:")
print(df4.columns.tolist())

print("\ng_location_disambiguated columns:")
print(df5.columns.tolist())

g_patent columns:
['patent_id', 'patent_type', 'patent_date', 'patent_title', 'wipo_kind', 'num_claims', 'withdrawn', 'filename']

g_patent_abstract columns:
['patent_id', 'patent_abstract']

g_inventor_disambiguated columns:
['patent_id', 'inventor_sequence', 'inventor_id', 'disambig_inventor_name_first', 'disambig_inventor_name_last', 'gender_code', 'location_id']

g_assignee_disambiguated columns:
['patent_id', 'assignee_sequence', 'assignee_id', 'disambig_assignee_individual_name_first', 'disambig_assignee_individual_name_last', 'disambig_assignee_organization', 'assignee_type', 'location_id']

g_location_disambiguated columns:
['location_id', 'disambig_city', 'disambig_state', 'disambig_country', 'latitude', 'longitude', 'county', 'state_fips', 'county_fips']


In [6]:


patent_df = pd.read_csv(
    "../data/extracted/g_patent.tsv",
    sep="\t",
    low_memory=False
)

abstract_df = pd.read_csv(
    "../data/extracted/g_patent_abstract.tsv",
    sep="\t",
    low_memory=False
)

inventor_df = pd.read_csv(
    "../data/extracted/g_inventor_disambiguated.tsv",
    sep="\t",
    low_memory=False
)

assignee_df = pd.read_csv(
    "../data/extracted/g_assignee_disambiguated.tsv",
    sep="\t",
    low_memory=False
)

location_df = pd.read_csv(
    "../data/extracted/g_location_disambiguated.tsv",
    sep="\t",
    low_memory=False
)



print("files loaded okay")
print("patent_df shape:", patent_df.shape)
print("abstract_df shape:", abstract_df.shape)
print("inventor_df shape:", inventor_df.shape)
print("assignee_df shape:", assignee_df.shape)
print("location_df shape:", location_df.shape)

files loaded okay
patent_df shape: (9454161, 8)
abstract_df shape: (9454161, 2)
inventor_df shape: (24037380, 7)
assignee_df shape: (8751310, 8)
location_df shape: (100452, 9)


In [7]:
# making the patents table
# keeping only the columns i actually need for the project

patents = patent_df[[
    "patent_id",
    "patent_title",
    "patent_date"
]].copy()

abstracts = abstract_df[[
    "patent_id",
    "patent_abstract"
]].copy()

# joining patent info with abstract using patent_id
patents = patents.merge(abstracts, on="patent_id", how="left")

# renaming columns so they look cleaner and match the assignment better
patents = patents.rename(columns={
    "patent_title": "title",
    "patent_date": "filing_date",
    "patent_abstract": "abstract"
})

# fixing the date column and pulling out the year
patents["filing_date"] = pd.to_datetime(patents["filing_date"], errors="coerce")
patents["year"] = patents["filing_date"].dt.year

# filling empty abstracts so the table doesnt look messy
patents["abstract"] = patents["abstract"].fillna("")

# removing duplicate patents just in case
patents = patents.drop_duplicates(subset=["patent_id"])

print("patents table done")
print(patents.head())
print("shape:", patents.shape)

patents table done
  patent_id                                              title filing_date  \
0  10000000  Coherent LADAR using intra-pixel quadrature de...  2018-06-19   
1  10000001  Injection molding machine and mold thickness c...  2018-06-19   
2  10000002  Method for manufacturing polymer film and co-e...  2018-06-19   
3  10000003  Method for producing a container from a thermo...  2018-06-19   
4  10000004  Process of obtaining a double-oriented film, c...  2018-06-19   

                                            abstract  year  
0  A frequency modulated (coherent) laser detecti...  2018  
1  The injection molding machine includes a fixed...  2018  
2  The present invention relates to: a method for...  2018  
3  The invention relates to a method for producin...  2018  
4  The present invention relates to provides a do...  2018  
shape: (9454161, 5)


In [8]:
# making the inventors table

inventors = inventor_df[[
    "inventor_id",
    "disambig_inventor_name_first",
    "disambig_inventor_name_last",
    "location_id"
]].copy()

# joining first and last name together
inventors["name"] = (
    inventors["disambig_inventor_name_first"].fillna("").astype(str).str.strip()
    + " " +
    inventors["disambig_inventor_name_last"].fillna("").astype(str).str.strip()
).str.strip()

# merging with location table to get country
inventors = inventors.merge(
    location_df[["location_id", "disambig_country"]],
    on="location_id",
    how="left"
)

# renaming the country column so it looks cleaner
inventors = inventors.rename(columns={
    "disambig_country": "country"
})

inventors["country"] = inventors["country"].fillna("Unknown")

inventors = inventors[[
    "inventor_id",
    "name",
    "country",
    "location_id"
]]

inventors = inventors.drop_duplicates(subset=["inventor_id"])

print("inventors table done")
print(inventors.head())
print("shape:", inventors.shape)

inventors table done
            inventor_id                  name  country  \
0    fl:we_ln:jiang-165         Wenjing Jiang       CN   
1    fl:ei_ln:baumker-1          Eiko BÄUMKER       DE   
2    fl:ri_ln:kroeger-1       Richard Kroeger  Unknown   
3       fl:th_ln:bush-1        Thomas A. Bush  Unknown   
4  fl:ma_ln:boudreaux-4  Matthew F. Boudreaux       US   

                            location_id  
0  9d072d42-49af-11ed-9879-1234bde3cd05  
1  67149e17-49af-11ed-9879-1234bde3cd05  
2                                   NaN  
3                                   NaN  
4  04726932-16c8-11ed-9b5f-1234bde3cd05  
shape: (4294034, 4)


In [9]:
# making companies table and also adding country from location file

companies = assignee_df[[
    "assignee_id",
    "disambig_assignee_organization",
    "location_id"
]].copy()

companies = companies.rename(columns={
    "assignee_id": "company_id",
    "disambig_assignee_organization": "name"
})

# cleaning names a bit
companies["name"] = companies["name"].fillna("").astype(str).str.strip()

# removing blank company names coz they wont help in analysis
companies = companies[companies["name"] != ""]

# merging with location table to get company country too
companies = companies.merge(
    location_df[["location_id", "disambig_country"]],
    on="location_id",
    how="left"
)

companies = companies.rename(columns={
    "disambig_country": "country"
})

companies["country"] = companies["country"].fillna("Unknown")

companies = companies[[
    "company_id",
    "name",
    "country",
    "location_id"
]]

companies = companies.drop_duplicates(subset=["company_id"])

print("companies table done")
print(companies.head())
print("shape:", companies.shape)

companies table done
                             company_id  \
0  a9d256f6-26f6-4553-a8ae-1a1bf4de1030   
1  3ffdc6b4-9949-4395-9aae-f90cb3a18637   
2  bdcc4e9e-662e-4d11-bc1a-7beddcb4e431   
3  4840cedc-7dcd-4aa9-9d62-ee877f7ef128   
4  45778076-46c5-4875-ad40-97b3a381a155   

                                                name country  \
0                            Metal Works Ramat David      IL   
1                       DIVERGENT TECHNOLOGIES, INC.      US   
2                           U.S. Philips Corporation      US   
3                                  Xerox Corporation      US   
4  Commonwealth Scientific and Industrial Researc...      AU   

                            location_id  
0  50dc5d46-16c8-11ed-9b5f-1234bde3cd05  
1  15c69712-16c8-11ed-9b5f-1234bde3cd05  
2  92237ca2-16c8-11ed-9b5f-1234bde3cd05  
3  0cd1998f-16c8-11ed-9b5f-1234bde3cd05  
4  4d36742f-16c8-11ed-9b5f-1234bde3cd05  
shape: (516032, 4)


In [10]:
# now making the relationships table
# this one connects patent -> inventor -> company
# not super perfect for real life but very okay for this project

patent_inventor = inventor_df[[
    "patent_id",
    "inventor_id"
]].copy()

patent_company = assignee_df[[
    "patent_id",
    "assignee_id"
]].copy()

patent_company = patent_company.rename(columns={
    "assignee_id": "company_id"
})

relationships = patent_inventor.merge(patent_company, on="patent_id", how="left")

relationships = relationships.drop_duplicates()

print("relationships table done")
print(relationships.head())
print("shape:", relationships.shape)

relationships table done
  patent_id           inventor_id                            company_id
0  D1006496    fl:we_ln:jiang-165                                   NaN
1  12029253    fl:ei_ln:baumker-1  41946e18-f5c5-4724-a6ee-fc0a425d198a
2   6584128    fl:ri_ln:kroeger-1  38969e56-ae8b-409d-9dd9-f67b77acb42b
3   4789863       fl:th_ln:bush-1                                   NaN
4  11161990  fl:ma_ln:boudreaux-4  92aa0a22-a0d8-4ffc-9026-aeda00ae190c
shape: (25291173, 3)


In [11]:
# saving all the clean tables as csv files
# this is what i will later load into sqlite

patents.to_csv("../data/clean_patents.csv", index=False)
inventors.to_csv("../data/clean_inventors.csv", index=False)
companies.to_csv("../data/clean_companies.csv", index=False)
relationships.to_csv("../data/clean_relationships.csv", index=False)


print("all clean csv files saved succesfully")

all clean csv files saved succesfully


In [12]:
print("clean_patents:", patents.shape)
print("clean_inventors:", inventors.shape)
print("clean_companies:", companies.shape)
print("clean_relationships:", relationships.shape)

clean_patents: (9454161, 5)
clean_inventors: (4294034, 4)
clean_companies: (516032, 4)
clean_relationships: (25291173, 3)
